# PROVA NEURAL NETWORK IN SIMBIOSI CON SKLEARN

---

Leggo il nostro dict in `.pkl`

In [1]:
import pickle as pkl

with open('../data/ottani_NIR.pkl', 'rb') as f:
    dizionario = pkl.load(f)
    
# dati train
train_ones = dizionario['train']['NIR']['ones']
train_zeros = dizionario['train']['NIR']['zeros']
train_labels = dizionario['train']['labels']

# dati test
test_ones = dizionario['test']['NIR']['ones']
test_zeros = dizionario['test']['NIR']['zeros']
test_labels = dizionario['test']['labels']


In [2]:
import numpy as np

train_x = np.concatenate((train_zeros, train_ones))
test_x = np.concatenate((test_zeros,test_ones))


importo neural network

In [ ]:
import sys
sys.path.append("../../ML_app")

from ML_app.single_neuron import single_neuron as neural_network

---

## SMOOTHING 

provo a filtrare i dati con Savitzky-Golay

In [4]:
from scipy.signal import savgol_filter

window_size = 10
poly_order = 3

for i in range(train_x.shape[0]):
    train_x[i] = savgol_filter(train_x[i], window_size, poly_order)

for j in range(test_x.shape[0]):
    test_x[j] = savgol_filter(test_x[j], window_size, poly_order)

---

In [5]:
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

from sklearn.ensemble import RandomForestClassifier as RFC

In [ ]:
rkf = RepeatedStratifiedKFold(n_splits=8, n_repeats=10, random_state=42)

# SCALING
SCALER_OPTIONS = [MinMaxScaler()]
# PCA
N_COMPONENTS_OPTIONS = [2, 7, 19, 37, None]
# ESTIMATOR
LEARNING_RATE_OPTIONS = [0.5, 0.9, 0.95]
EPOCHE_OPTIONS = [1_000, 2_000, 10_000, 20_000]
LOSS_FUNCTION_OPTIONS = ['MSE']

# 1. Definizione Pipeline #
pipe = Pipeline([
    # Step 1: Scaling
    ("scaling", MinMaxScaler()),       
    
    # Step 2: Riduzione dimensionalità (PCA)
    ("reduce_dim", PCA(random_state=None)),
    
    # Step 3: Classificatore
    ("classify", neural_network(random_state=None, epoche=10_000)) 
])

# 2. Definizione griglia dei parametri #
param_grid = [
{
    # Per provare diversi scaler
    "scaling": SCALER_OPTIONS,
    
    # Per provare diversi numeri di componenti (PCA)
    "reduce_dim__n_components": N_COMPONENTS_OPTIONS, 
    
    # Per cambiare parametri del neurone
    "classify__epoche": EPOCHE_OPTIONS,
    "classify__nn_learning_rate": LEARNING_RATE_OPTIONS,
    
},
{
    # Per provare diversi scaler
    "scaling": SCALER_OPTIONS,
    
    # Per saltare la riduzione
    "reduce_dim": ['passthrough'], 
    
    # Per cambiare parametri del neurone
    "classify__epoche": EPOCHE_OPTIONS,
    "classify__nn_learning_rate": LEARNING_RATE_OPTIONS,
}]

# 3. Configurazione GridSearch #
grid = GridSearchCV(
    pipe, 
    param_grid=param_grid, 
    cv=rkf,
    n_jobs=-1, # «Number of jobs to run in parallel. -1 means using all processors»
    scoring={
        'score': 'accuracy',
        'sensitivity': 'recall'  # recall = sensitivity
    },
    refit='score', # «For multiple metric evaluation, needs to be a str denoting the
    # scorer to use to find the best parameters for refitting the estimator at the end»
    return_train_score=False
)

# 4. Training e Validation (su Segnale B) #
grid.fit(train_x, train_labels)

# 5. Risultati #
print(f"La miglior configurazione: {grid.best_params_}")
print(f"Fornisce accuracy in validation: {grid.best_score_:.4f}")

# 6. Test su segnale A #
accuracy_finale = grid.score(test_x, test_labels)
print(f"Risultato sul set indipendente: {accuracy_finale:.4f}")

La miglior configurazione: {'classify__epoche': 20000, 'classify__nn_learning_rate': 0.9, 'reduce_dim__n_components': 37, 'scaling': MinMaxScaler()}
Fornisce accuracy in validation: 0.9458
Risultato sul set indipendente: 0.9167


In [7]:
import pandas as pd
# Conversione dei risultati in DataFrame
results_df = pd.DataFrame(grid.cv_results_)
results_df.to_pickle("results_nn_GridSearch.pkl")

In [8]:
import pandas as pd

results_df = pd.DataFrame(grid.cv_results_)
# Ciascuna combinazione di parametri è una riga
print(f"Numero totale di configurazioni provate: {results_df.shape[0]}")

# Selezioniamo solo le colonne interessanti per pulire la vista
columns_to_show = [
    'param_scaling',
    'param_reduce_dim__n_components', 
    'param_classify__epoche',
    'param_classify__nn_learning_rate', 
    'mean_test_score', 
    'std_test_score', 
    'mean_test_sensitivity',
    'rank_test_score'
]

# Ordiniamo per classifica (rank_test_score)
analysis = results_df[columns_to_show].sort_values('rank_test_score')

# Se si ha, è comodo aprire analysis in un viewer tipo Data Wrangler
analysis.head(20)

Numero totale di configurazioni provate: 72


,param_scaling,param_reduce_dim__n_components,param_classify__epoche,param_classify__nn_learning_rate,mean_test_score,std_test_score,mean_test_sensitivity,rank_test_score
53,MinMaxScaler(),37,20000,0.90,0.945833,0.082390,0.995833,1
58,MinMaxScaler(),37,20000,0.95,0.945833,0.082390,0.995833,1
57,MinMaxScaler(),19,20000,0.95,0.943750,0.087177,0.995833,3
34,MinMaxScaler(),None,10000,0.50,0.943750,0.083099,0.995833,3
32,MinMaxScaler(),19,10000,0.50,0.943750,0.087177,0.995833,3
38,MinMaxScaler(),37,10000,0.90,0.943750,0.087177,0.995833,3
39,MinMaxScaler(),None,10000,0.90,0.943750,0.087177,0.995833,3
28,MinMaxScaler(),37,2000,0.95,0.943750,0.087177,0.995833,3
42,MinMaxScaler(),19,10000,0.95,0.943750,0.087177,0.995833,3
54,MinMaxScaler(),None,20000,0.90,0.943750,0.087177,0.995833,3


In [ ]:
analysis